In [15]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [26]:
load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AQ') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'

gemini = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

API key looks good so far


In [27]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [28]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [29]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [30]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [31]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [32]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [33]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'company info', 'url': 'https://huggingface.co'},
  {'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'products page', 'url': 'https://huggingface.co/models'},
  {'type': 'products page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'products page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'products page', 'url': 'https://huggingface.co/storage'},
  {'type': 'products page', 'url': 'https://huggingface.co/tasks'},
  {'type': 'products page', 'url': 'https://huggingface.co/chat'},
  {'type': 'products page', 'url': 'https://huggingface.co/hardware'},
  {'type': 'products page', 'url': 'https://huggingface.co/pro'},
  {'type': 'prod

In [34]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [35]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 32 relevant links


{'links': [{'type': 'Home page', 'url': 'https://huggingface.co'},
  {'type': 'About us', 'url': 'https://huggingface.co/learn'},
  {'type': 'Brand information', 'url': 'https://huggingface.co/brand'},
  {'type': 'Blog / News', 'url': 'https://huggingface.co/blog'},
  {'type': 'Latest posts', 'url': 'https://huggingface.co/posts'},
  {'type': 'Research papers', 'url': 'https://huggingface.co/papers'},
  {'type': 'Product changelog', 'url': 'https://huggingface.co/changelog'},
  {'type': 'Careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'Models Hub', 'url': 'https://huggingface.co/models'},
  {'type': 'Datasets Hub', 'url': 'https://huggingface.co/datasets'},
  {'type': 'Spaces', 'url': 'https://huggingface.co/spaces'},
  {'type': 'Enterprise solutions', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'Pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'Pro offerings', 'url': 'https://huggingface.co/pro'},
  {'type': 'Inference API', 'url'

In [36]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [37]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 24 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
tencent/Hy3
Updated
4 days ago
•
6.92k
•
654
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
12 days ago
•
1.91M
•
1.96k
zai-org/GLM-5.2
Updated
8 days ago
•
393k
•
3.77k
InternScience/Agents-A1
Updated
1 day ago
•
25.8k
•
460
baidu/Unlimited-OCR
Updated
8 days ago
•
1.32M


In [38]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [41]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:2_000] # Truncate if more than 5,000 characters
    return user_prompt

In [42]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 31 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ntencent/Hy3\nUpdated\n4 days ago\n•\n6.92k\n•\n654\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n12 days a

In [43]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [44]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 25 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


**Hugging Face: The AI Community Building the Future**

Hugging Face is the premier platform where the global machine learning community collaborates to build the future of artificial intelligence. We are dedicated to fostering innovation, making state-of-the-art AI accessible, and driving progress through an open, collaborative ecosystem.

**Our Core Offerings:**

*   **Models:** Explore and contribute to an expansive hub featuring over 2 million pre-trained machine learning models, ready for diverse applications.
*   **Datasets:** Access and share a rich collection of over 500,000 datasets, essential for training and evaluating AI systems.
*   **Spaces (AI Applications):** Discover, run, and host over 1 million interactive AI applications, showcasing cutting-edge research and practical tools.
*   **Comprehensive Tools:** Beyond our core assets, we provide essential tools like Storage Buckets, Inference Endpoints, documentation, HuggingChat, and robust enterprise solutions.

**A Collaborative Community at Heart:**

Collaboration is the driving force behind Hugging Face. We cultivate a vibrant and active community where developers, researchers, and organizations connect, share knowledge, and collectively advance AI. Our platforms encourage open contribution, learning, and engagement through resources like our Blog, Daily Papers, and community channels including Discord, Forum, and GitHub.

**Solutions for Every Innovator:**

Whether you are an individual developer, a research institution, or a large enterprise, Hugging Face provides tailored solutions. We offer advanced infrastructure, enterprise support, and specialized services like Hugging Face PRO to meet diverse needs, helping users scale their AI ambitions.

**Join Us in Shaping AI's Tomorrow:**

Hugging Face is more than just a platform; it's a movement towards an open and collaborative AI future. We invite you to explore our ecosystem, contribute to our community, and leverage our resources to build the next generation of artificial intelligence.

*Note: Specific details regarding company culture beyond collaboration, and explicit career/job opportunities were not available in the provided text.*

In [45]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [46]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 18 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the leading platform where the global machine learning community collaborates to build the future of AI. We are an open-source pioneer dedicated to democratizing AI, providing the tools and infrastructure for researchers, developers, and enterprises to innovate together.

## What We Do

We provide a comprehensive ecosystem for AI development, fostering collaboration across various domains:

*   **Models:** Explore and utilize over 2 million pre-trained models, ranging from natural language processing to computer vision and beyond. Our platform facilitates sharing, discovery, and application of state-of-the-art AI models.
*   **Datasets:** Access and contribute to over 500,000 datasets, essential for training and evaluating AI models. We promote data sharing and standardization to accelerate research and development.
*   **Spaces:** Deploy and showcase your AI applications with ease. Our Spaces offer a collaborative environment to build and share interactive demos, with over 1 million applications available.
*   **Buckets:** Securely store and manage your machine learning artifacts, ensuring seamless integration with your workflows.

## Our Community & Culture

At the heart of Hugging Face is a vibrant and active community. We believe in open collaboration, continuous learning, and pushing the boundaries of what AI can achieve. Our community engages through:

*   **HuggingChat:** For real-time discussions and support.
*   **Discord & Forum:** Platforms for detailed conversations, troubleshooting, and knowledge sharing.
*   **GitHub:** For open-source contributions and project collaboration.
*   **Blog & Daily Papers:** Stay informed with the latest AI research, trends, and Hugging Face updates.
*   **Learn:** Educational resources to help you master AI concepts and tools.

We foster a culture of innovation, transparency, and accessibility, enabling everyone from individual researchers to large organizations to participate in the AI revolution.

## For Our Customers (Team & Enterprise)

Hugging Face offers robust solutions tailored for professional teams and enterprises, designed to accelerate AI adoption and deployment:

*   **Hugging Face PRO:** Enhanced features and support for professional users.
*   **Enterprise Support:** Dedicated assistance to ensure your AI projects run smoothly.
*   **Inference Providers & Endpoints:** Scalable and efficient infrastructure for deploying your AI models in production.
*   **Storage Buckets:** Secure and reliable storage solutions for enterprise-grade ML assets.

We empower businesses to leverage cutting-edge AI, offering the reliability and performance needed for real-world applications.

## Careers at Hugging Face

Join a passionate team dedicated to building the future of AI. We are constantly seeking talented individuals who are eager to contribute to an open and collaborative environment. If you are enthusiastic about machine learning, open-source, and making a global impact, explore opportunities to join our growing team. We value innovation, expertise, and a commitment to democratizing AI.

**Hugging Face: Empowering the world to build, train, and deploy AI.**